## 0. Setup local / Colab project root

**VI:** Cell này mount Google Drive và tìm project root để notebook chạy được cả local lẫn Google Colab.  
Nếu Colab không tìm thấy project tự động, kiểm tra `PROJECT_ROOT_OVERRIDE` bên dưới.  

**EN:** Mounts Google Drive and locates the project root for local and Colab.  
If auto-detection fails, set `PROJECT_ROOT_OVERRIDE` to your Drive path manually.

In [4]:
from pathlib import Path
import os
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
MOUNT_GOOGLE_DRIVE    = IN_COLAB
# ── Đặt lại nếu Colab không tự detect được project ──────────────────────────
PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive" if IN_COLAB else ""
SEARCH_GOOGLE_DRIVE   = IN_COLAB
MAX_SEARCH_DIRS       = 4000

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


def _looks_like_project_root(path: Path) -> bool:
    return (path / "notebooks").exists() and (path / "requirements.txt").exists()


def _walk_for_project_roots(base: Path, max_dirs: int = MAX_SEARCH_DIRS) -> list:
    if not base.exists():
        return []
    matches, visited = [], 0
    skip = {".git", "__pycache__", ".ipynb_checkpoints", "data", "artifacts", "results", "reports"}
    for current, dirs, _ in os.walk(base):
        visited += 1
        if visited > max_dirs:
            break
        dirs[:] = [d for d in dirs if d not in skip]
        if _looks_like_project_root(Path(current)):
            matches.append(Path(current))
    return matches


if PROJECT_ROOT_OVERRIDE:
    _PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif _looks_like_project_root(Path.cwd()):
    _PROJECT_ROOT = Path.cwd().resolve()
elif _looks_like_project_root(Path.cwd().parent):
    _PROJECT_ROOT = Path.cwd().parent.resolve()
elif IN_COLAB and SEARCH_GOOGLE_DRIVE:
    _candidates = _walk_for_project_roots(Path("/content/drive/MyDrive"))
    if not _candidates:
        raise FileNotFoundError(
            "Could not find project root in Google Drive. "
            "Set PROJECT_ROOT_OVERRIDE manually."
        )
    _PROJECT_ROOT = _candidates[0].resolve()
else:
    raise FileNotFoundError(
        "Could not find project root. "
        "Run this notebook from the project folder or set PROJECT_ROOT_OVERRIDE."
    )

os.chdir(_PROJECT_ROOT)
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

print("IN_COLAB     :", IN_COLAB)
print("PROJECT_ROOT :", _PROJECT_ROOT)

Mounted at /content/drive
IN_COLAB     : True
PROJECT_ROOT : /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive


# Phase 3 — LLM Representation and Cost-Sensitive Q-value Contextual Bandit

**Upgraded 2026-05-27** — SGDClassifier → Gym-style Q-value Contextual Bandit.  
**RAM target: ≤ 12 GB** (local / Colab Free).

### Ba variants ablation
| Variant | Tên | Loại |
|---|---|---|
| 1 | `cost_sensitive_supervised_sgd` | Strong supervised baseline — không phải RL chính |
| 2 | `rl_q_bandit_without_embedding` | Q-value bandit — tabular features only |
| 3 | `rl_q_bandit_with_embedding` | Q-value bandit — tabular + MiniLM embedding |

### Chiến lược RAM
- `usecols` khi load CSV — chỉ đọc cột cần thiết.
- Serialize text theo **batch 50 k rows** — không giữ 3 Series đầy đủ.
- `del + gc.collect()` sau mỗi bước lớn.
- `float32` toàn bộ numeric arrays.
- `SGDRegressor` nhận sparse matrix natively — Variant 2 **không** to_dense.
- Variant 3: state = sparse_row ∥ emb_row **on-the-fly** trong env — không lưu `X_train_emb` đầy đủ.
- `predict_scores_batched` — evaluation theo batch, không giữ toàn bộ scores trong RAM.
- Chỉ **1 policy** trong RAM tại mỗi thời điểm.

Mặc định hiện tại `RUN_MODE = "sample_100k"`. Nếu 100k chạy ổn, tăng dần sang `"sample_200k"`, `"sample_300k"`; chỉ dùng `"full"` khi RAM/runtime cho phép.


## 1. Setup and Configuration

In [5]:
import gc
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safer for batch runs
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier, SGDRegressor
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

# ── CONFIG ────────────────────────────────────────────────────────────────────
RUN_MODE = "sample_100k"  # "smoke", "sample_100k", "sample_200k", "sample_300k", or "full"
RUN_MODE_SAMPLE_ROWS = {
    "smoke": 10_000,
    "sample_100k": 100_000,
    "sample_200k": 200_000,
    "sample_300k": 300_000,
    "full": None,
}
if RUN_MODE not in RUN_MODE_SAMPLE_ROWS:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE!r}. Use one of {list(RUN_MODE_SAMPLE_ROWS)}.")

SAMPLE_ROWS  = RUN_MODE_SAMPLE_ROWS[RUN_MODE]
SAMPLE_ROWS_LABEL = SAMPLE_ROWS if SAMPLE_ROWS is not None else "full"
RUN_OUTPUT_TAG = RUN_MODE
RANDOM_STATE = 42
N_JOBS       = 1           # fixed to 1 for 12 GB stability

EMBEDDING_MODEL_NAME               = "sentence-transformers/all-MiniLM-L6-v2"
AUTO_INSTALL_SENTENCE_TRANSFORMERS = IN_COLAB
EMBEDDING_BATCH_SIZE               = 32 if RUN_MODE.startswith("sample_") else (64 if RUN_MODE == "smoke" else 128)
EMBEDDING_REDUCTION_COMPONENTS     = 64
TEXT_SERIALIZE_CHUNK               = 50_000   # batch size for text serialization
SCORE_BATCH_SIZE                   = 4_096    # batch size for predict_scores

# ── RL hyper-params ───────────────────────────────────────────────────────────
RL_N_EPOCHS      = 3 if RUN_MODE == "smoke" or RUN_MODE.startswith("sample_") else 5
RL_EPSILON_START = 0.30
RL_EPSILON_END   = 0.05
RL_LR            = 0.005 if RUN_MODE.startswith("sample_") else 0.01

# Reward scaling affects Q-bandit training targets only. Business-cost evaluation remains raw.
REWARD_SCALE_MODE = "p95_scaled" if RUN_MODE.startswith("sample_") else "raw"  # "raw", "p95_scaled", "log1p"
SHUFFLE_TRAIN_EACH_EPOCH = False  # keep TransactionDT order by default; set True only for sensitivity analysis.
RUN_Q_BANDIT_TUNING_GRID = False  # heavy grid is intentionally opt-in.
Q_BANDIT_TUNING_GRID = [
    {"rl_lr": 0.001, "epochs": 3, "eps_start": 0.30, "eps_end": 0.05, "reward_scale_mode": "p95_scaled"},
    {"rl_lr": 0.005, "epochs": 3, "eps_start": 0.30, "eps_end": 0.05, "reward_scale_mode": "p95_scaled"},
    {"rl_lr": 0.005, "epochs": 5, "eps_start": 0.50, "eps_end": 0.05, "reward_scale_mode": "p95_scaled"},
]

# ── Paths (relative to PROJECT_ROOT resolved in Cell 0) ───────────────────────
PROJECT_ROOT   = _PROJECT_ROOT
RAW_DIR        = PROJECT_ROOT / "data"      / "raw"
RESULTS_DIR    = PROJECT_ROOT / "results"
FIGURES_DIR    = PROJECT_ROOT / "reports"   / "figures"
EMBEDDINGS_DIR = PROJECT_ROOT / "artifacts" / "embeddings"

for _d in [RAW_DIR, RESULTS_DIR, FIGURES_DIR, EMBEDDINGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

COST_CONFIGS = {
    "Cost-A": {"alpha": 0.05, "beta": 1.00},
    "Cost-B": {"alpha": 0.10, "beta": 2.00},
    "Cost-C": {"alpha": 0.20, "beta": 5.00},
}

print(f"RUN_MODE={RUN_MODE} | SAMPLE_ROWS={SAMPLE_ROWS_LABEL} | RUN_OUTPUT_TAG={RUN_OUTPUT_TAG} | N_JOBS={N_JOBS}")
print(f"RL epochs={RL_N_EPOCHS} | eps {RL_EPSILON_START}→{RL_EPSILON_END} | lr={RL_LR}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Python {sys.version.split()[0]} | {platform.platform()}")


RUN_MODE=sample_100k | SAMPLE_ROWS=100000 | RUN_OUTPUT_TAG=sample_100k | N_JOBS=1
RL epochs=3 | eps 0.3→0.05 | lr=0.005
PROJECT_ROOT : /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive
Python 3.12.13 | Linux-6.6.122+-x86_64-with-glibc2.35


## 2. Load, Join, and Time-Based Split

**RAM key:** `usecols` khi đọc CSV — chỉ load cột thực sự cần.  
Identity CSV nhỏ → load toàn bộ, giữ chỉ cột cần sau merge.  
Sau join: `del` transaction + identity DataFrames ngay.

In [6]:
REQUIRED_RAW_FILES = ["train_transaction.csv", "train_identity.csv"]

# ── Column selection (RAM optimization: usecols) ───────────────────────────────
_TXN_ALWAYS = ["TransactionID", "TransactionDT", "isFraud", "TransactionAmt"]
_TXN_NUMERIC = [
    "addr1", "addr2", "dist1", "dist2",
    *[f"C{i}" for i in range(1, 15)],
    *[f"D{i}" for i in range(1, 16)],
]
_TXN_CATEGORICAL = [
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "P_emaildomain", "R_emaildomain",
]
TRANSACTION_COLS_NEEDED = set(_TXN_ALWAYS + _TXN_NUMERIC + _TXN_CATEGORICAL)
IDENTITY_COLS_NEEDED    = None   # small file — load all


def stage_colab_raw(raw_dir: Path) -> None:
    raw_dir.mkdir(parents=True, exist_ok=True)
    if all((raw_dir / n).exists() for n in REQUIRED_RAW_FILES):
        return
    for search_root in [
        Path("/content/ieee-fraud-detection"),
        Path("/content/drive/MyDrive/ieee-fraud-detection"),
        Path("/content/drive/MyDrive/Kaggle/ieee-fraud-detection"),
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]:
        if not search_root.exists():
            continue
        for name in REQUIRED_RAW_FILES:
            target = raw_dir / name
            if not target.exists():
                hits = list(search_root.rglob(name))[:3]
                if hits:
                    shutil.copy2(hits[0], target)


def validate_raw(raw_dir: Path) -> None:
    missing = [n for n in REQUIRED_RAW_FILES if not (raw_dir / n).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing IEEE-CIS files: {', '.join(missing)}. "
            f"Put them in {raw_dir} or upload to Colab/Drive."
        )


def load_ieee_data(raw_dir: Path, sample_rows=None):
    txn_path = raw_dir / "train_transaction.csv"
    idn_path = raw_dir / "train_identity.csv"
    txn_header = pd.read_csv(txn_path, nrows=0).columns.tolist()
    usecols = [c for c in txn_header if c in TRANSACTION_COLS_NEEDED]
    print(f"  Transaction CSV: loading {len(usecols)}/{len(txn_header)} columns.")
    txn = pd.read_csv(txn_path, nrows=sample_rows, usecols=usecols)
    idn = pd.read_csv(idn_path)
    joined = txn.merge(idn, on="TransactionID", how="left", suffixes=("", "_identity"))
    assert len(joined) == len(txn)
    assert "isFraud" in joined.columns
    del txn, idn
    gc.collect()
    return joined


def split_by_transaction_dt(df: pd.DataFrame, ratios=(0.70, 0.15, 0.15)):
    assert math.isclose(sum(ratios), 1.0, rel_tol=1e-9)
    ordered = df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)
    n  = len(ordered)
    i1 = int(n * ratios[0])
    i2 = i1 + int(n * ratios[1])
    train = ordered.iloc[:i1].copy()
    val   = ordered.iloc[i1:i2].copy()
    test  = ordered.iloc[i2:].copy()
    del ordered
    gc.collect()
    s_tr = set(train["TransactionID"])
    s_va = set(val["TransactionID"])
    s_te = set(test["TransactionID"])
    assert not (s_tr & s_va or s_tr & s_te or s_va & s_te), "TransactionID overlap"
    return train, val, test


stage_colab_raw(RAW_DIR)
validate_raw(RAW_DIR)

print("Loading data...")
joined_df = load_ieee_data(RAW_DIR, sample_rows=SAMPLE_ROWS)
train_df, validation_df, test_df = split_by_transaction_dt(joined_df)
del joined_df
gc.collect()

splits = {"train": train_df, "validation": validation_df, "test": test_df}
split_summary = pd.DataFrame([
    {"run_mode": RUN_MODE, "sample_rows": SAMPLE_ROWS_LABEL,
     "split": k, "rows": len(v), "fraud": int(v["isFraud"].sum()),
     "fraud_rate": float(v["isFraud"].mean())}
    for k, v in splits.items()
])
split_summary.to_csv(RESULTS_DIR / "phase3_split_summary.csv", index=False)
split_summary.to_csv(RESULTS_DIR / f"phase3_split_summary_{RUN_OUTPUT_TAG}.csv", index=False)
display(split_summary)
if RUN_MODE == "smoke":
    print("[SMOKE] Results are pipeline verification only, not final claims.")
elif RUN_MODE.startswith("sample_"):
    print(f"[{RUN_MODE.upper()}] Use only with Phase 2 {RUN_MODE} results for matched comparison.")


Loading data...
  Transaction CSV: loading 46/394 columns.


,run_mode,sample_rows,split,rows,fraud,fraud_rate
0,sample_100k,100000,train,70000,1879,0.026843
1,sample_100k,100000,validation,15000,377,0.025133
2,sample_100k,100000,test,15000,305,0.020333


[SAMPLE_100K] Use only with Phase 2 sample_100k results for matched comparison.


## 3. Feature Policy and Core Arrays

In [7]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    s = out["TransactionDT"].astype("float32")
    out["TransactionDT_log1p"]    = np.log1p(s).astype("float32")
    out["TransactionDT_mod_day"]  = (s % 86400).astype("float32")
    out["TransactionDT_mod_week"] = (s % (86400 * 7)).astype("float32")
    return out


TABULAR_NUMERIC_CANDIDATES = [
    "TransactionAmt", "addr1", "addr2", "dist1", "dist2",
    *[f"C{i}" for i in range(1, 15)],
    *[f"D{i}" for i in range(1, 16)],
    "TransactionDT_log1p", "TransactionDT_mod_day", "TransactionDT_mod_week",
]
TABULAR_CATEGORICAL_CANDIDATES = [
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "id_12", "id_15", "id_16", "id_28", "id_29",
    "id_30", "id_31", "id_34", "id_35", "id_36", "id_37", "id_38",
]
TEXT_FEATURE_COLUMNS = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "id_12", "id_15", "id_30", "id_31", "id_34",
]

train_features_df      = add_time_features(train_df)
validation_features_df = add_time_features(validation_df)
test_features_df       = add_time_features(test_df)

numeric_cols     = [c for c in TABULAR_NUMERIC_CANDIDATES     if c in train_features_df.columns]
categorical_cols = [c for c in TABULAR_CATEGORICAL_CANDIDATES if c in train_features_df.columns]
text_cols        = [c for c in TEXT_FEATURE_COLUMNS           if c in train_df.columns]

for _f in ["TransactionID", "isFraud", "TransactionDT"]:
    assert _f not in numeric_cols and _f not in categorical_cols, f"Leak: {_f}"

y_train      = train_df["isFraud"].astype("int8").to_numpy()
y_validation = validation_df["isFraud"].astype("int8").to_numpy()
y_test       = test_df["isFraud"].astype("int8").to_numpy()

amount_train      = train_df["TransactionAmt"].fillna(0).astype("float32").to_numpy()
amount_validation = validation_df["TransactionAmt"].fillna(0).astype("float32").to_numpy()
amount_test       = test_df["TransactionAmt"].fillna(0).astype("float32").to_numpy()

id_train      = train_df["TransactionID"].astype(str).to_numpy()
id_validation = validation_df["TransactionID"].astype(str).to_numpy()
id_test       = test_df["TransactionID"].astype(str).to_numpy()

for _sname, _y in {"train": y_train, "val": y_validation, "test": y_test}.items():
    assert len(np.unique(_y)) >= 2, f"{_sname} split has only one class — increase SAMPLE_ROWS."

pd.DataFrame([
    {"item": "run_mode",                  "value": RUN_MODE},
    {"item": "sample_rows",               "value": SAMPLE_ROWS_LABEL},
    {"item": "run_output_tag",            "value": RUN_OUTPUT_TAG},
    {"item": "numeric_state_columns",     "value": len(numeric_cols)},
    {"item": "categorical_state_columns", "value": len(categorical_cols)},
    {"item": "text_columns",              "value": len(text_cols)},
    {"item": "embedding_model_target",    "value": EMBEDDING_MODEL_NAME},
]).to_csv(RESULTS_DIR / "phase3_feature_policy.csv", index=False)

print(f"numeric={len(numeric_cols)} | categorical={len(categorical_cols)} | text={len(text_cols)}")
print(f"y_train dtype={y_train.dtype} | amount_train dtype={amount_train.dtype}")


numeric=37 | categorical=23 | text=19
y_train dtype=int8 | amount_train dtype=float32


## 4. Neutral Table-to-Text Serialization

**RAM key:** Serialize theo **batch** (`TEXT_SERIALIZE_CHUNK` rows/lần).

In [8]:
FORBIDDEN_TEXT_TOKENS = [
    "isfraud", "fraud", "label", "target", "score", "prediction", "predict",
    "risk", "risky", "suspicious", "high risk", "low risk",
]
PRODUCT_INTERP_TOKENS = [
    "product category", "goods category", "service category", "merchandise",
]


def _clean(v) -> str:
    if pd.isna(v):
        return "unknown"
    if isinstance(v, float):
        return str(int(v)) if v.is_integer() else f"{v:.4g}"
    s = str(v).strip()
    return s if s else "unknown"


def serialize_transaction(row: pd.Series) -> str:
    g = lambda c: _clean(row[c]) if c in row.index else "unknown"
    return (
        f"A transaction with amount {g('TransactionAmt')}, product code {g('ProductCD')}, "
        f"card fields card1 {g('card1')}, card2 {g('card2')}, card3 {g('card3')}, "
        f"card4 {g('card4')}, card5 {g('card5')}, card6 {g('card6')}, "
        f"address fields addr1 {g('addr1')} and addr2 {g('addr2')}, "
        f"purchaser email domain {g('P_emaildomain')}, recipient email domain {g('R_emaildomain')}, "
        f"device type {g('DeviceType')}, device info {g('DeviceInfo')}, "
        f"identity fields id_12 {g('id_12')}, id_15 {g('id_15')}, "
        f"operating system {g('id_30')}, browser {g('id_31')}, screen {g('id_34')}."
    )


def serialize_split_batched(df: pd.DataFrame, chunk_size: int = TEXT_SERIALIZE_CHUNK) -> list:
    results = []
    n = len(df)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk = df.iloc[start:end][text_cols].apply(serialize_transaction, axis=1)
        results.extend(chunk.tolist())
        del chunk
    return results


def audit_texts(texts: list, split_name: str) -> dict:
    lower = [t.lower() for t in texts]
    forbidden = [tok for tok in FORBIDDEN_TEXT_TOKENS if any(tok in s for s in lower)]
    product   = [tok for tok in PRODUCT_INTERP_TOKENS  if any(tok in s for s in lower)]
    assert not forbidden and not product, (
        f"Text leakage FAILED [{split_name}]: forbidden={forbidden} product={product}"
    )
    return {"split": split_name, "rows": len(texts), "passed": True}


print("Serializing train text (batched)...")
train_texts = serialize_split_batched(train_df)
audit_tr = audit_texts(train_texts, "train")

print("Serializing validation text (batched)...")
val_texts = serialize_split_batched(validation_df)
audit_va = audit_texts(val_texts, "validation")

print("Serializing test text (batched)...")
test_texts = serialize_split_batched(test_df)
audit_te = audit_texts(test_texts, "test")

text_audit = pd.DataFrame([audit_tr, audit_va, audit_te])
text_audit.insert(0, "sample_rows", SAMPLE_ROWS_LABEL)
text_audit.insert(0, "run_mode", RUN_MODE)
text_audit.to_csv(RESULTS_DIR / "phase3_text_leakage_audit.csv", index=False)
text_audit.to_csv(RESULTS_DIR / f"phase3_text_leakage_audit_{RUN_OUTPUT_TAG}.csv", index=False)

_sample_n = min(5, len(train_texts))
pd.DataFrame({
    "run_mode": [RUN_MODE] * _sample_n,
    "sample_rows": [SAMPLE_ROWS_LABEL] * _sample_n,
    "split": ["train"] * _sample_n,
    "TransactionID": id_train[:_sample_n],
    "serialized_text": train_texts[:_sample_n],
}).to_csv(RESULTS_DIR / "phase3_serialized_text_sample.csv", index=False)

print("Text leakage audit: PASS")
display(text_audit)


Serializing train text (batched)...
Serializing validation text (batched)...
Serializing test text (batched)...
Text leakage audit: PASS


,run_mode,sample_rows,split,rows,passed
0,sample_100k,100000,train,70000,True
1,sample_100k,100000,validation,15000,True
2,sample_100k,100000,test,15000,True


## 5. Local Sentence Embedding and Cache

In [9]:
def try_import_sentence_transformer():
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer
    except ImportError:
        if AUTO_INSTALL_SENTENCE_TRANSFORMERS:
            print("Installing sentence-transformers for Colab...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
            from sentence_transformers import SentenceTransformer
            return SentenceTransformer
        return None


def emb_file(prefix, split):  return EMBEDDINGS_DIR / f"{prefix}_{split}.npy"
def meta_file(prefix):        return EMBEDDINGS_DIR / f"{prefix}_metadata.csv"


def cache_valid(prefix, counts):
    if not all(emb_file(prefix, s).exists() for s in counts): return False
    if not meta_file(prefix).exists(): return False
    try:
        meta = pd.read_csv(meta_file(prefix))
        return all(
            int(meta.loc[meta["split"] == s, "rows"].iloc[0]) == c
            for s, c in counts.items()
        )
    except Exception:
        return False


def encode_and_save_minilm(text_map, prefix, model):
    meta_rows = []
    for split in ["train", "validation", "test"]:
        texts = text_map[split]
        print(f"  Encoding {split} ({len(texts)} rows)...")
        mat = model.encode(
            texts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        np.save(emb_file(prefix, split), mat)
        meta_rows.append({
            "run_mode": RUN_MODE, "split": split,
            "rows": mat.shape[0], "embedding_dim": mat.shape[1],
            "backend": "minilm", "model_name": EMBEDDING_MODEL_NAME,
        })
        del mat, texts
        text_map[split] = None
        gc.collect()
    pd.DataFrame(meta_rows).to_csv(meta_file(prefix), index=False)
    return "minilm"


def encode_and_save_tfidf(text_map, prefix):
    print("  Using TF-IDF + TruncatedSVD fallback.")
    vect = TfidfVectorizer(max_features=2048, ngram_range=(1, 2), min_df=2)
    X_tr = vect.fit_transform(text_map["train"])
    n_comp = min(EMBEDDING_REDUCTION_COMPONENTS, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    meta_rows = []
    for split in ["train", "validation", "test"]:
        if split == "train":
            mat = svd.fit_transform(X_tr).astype(np.float32)
            del X_tr
        else:
            mat = svd.transform(vect.transform(text_map[split])).astype(np.float32)
        np.save(emb_file(prefix, split), mat)
        meta_rows.append({
            "run_mode": RUN_MODE, "split": split,
            "rows": mat.shape[0], "embedding_dim": mat.shape[1],
            "backend": "tfidf_svd_fallback",
            "model_name": "TF-IDF + TruncatedSVD fallback",
        })
        del mat
        text_map[split] = None
        gc.collect()
    del svd, vect
    pd.DataFrame(meta_rows).to_csv(meta_file(prefix), index=False)
    return "tfidf_svd_fallback"


def current_split_counts():
    global split_counts
    if all(name in globals() for name in ["train_texts", "val_texts", "test_texts"]):
        split_counts = {
            "train":      len(train_texts),
            "validation": len(val_texts),
            "test":       len(test_texts),
        }
        return split_counts
    if "split_counts" in globals():
        return split_counts
    raise RuntimeError(
        "Embedding split counts are unavailable. Run Section 4 (Neutral Table-to-Text Serialization) "
        "before Section 5/6."
    )


def load_or_create_raw_embeddings():
    global SentenceTransformerClass, embedding_prefix, embedding_backend
    global split_counts, emb_train_raw, emb_val_raw, emb_test_raw

    SentenceTransformerClass = try_import_sentence_transformer()
    if SentenceTransformerClass is not None:
        embedding_prefix  = f"{RUN_MODE}_minilm"
        embedding_backend = "minilm"
    else:
        embedding_prefix  = f"{RUN_MODE}_tfidf_svd_fallback"
        embedding_backend = "tfidf_svd_fallback"

    split_counts = current_split_counts()

    if cache_valid(embedding_prefix, split_counts):
        print(f"Loading cached embeddings [{embedding_prefix}]")
    else:
        missing_text_vars = [
            name for name in ["train_texts", "val_texts", "test_texts"]
            if name not in globals()
        ]
        if missing_text_vars:
            raise RuntimeError(
                "Embedding cache is missing or invalid and serialized text variables are unavailable: "
                f"{missing_text_vars}. Rerun Section 4 before Section 5."
            )
        text_map = {"train": train_texts, "validation": val_texts, "test": test_texts}
        if SentenceTransformerClass is not None:
            _model = SentenceTransformerClass(EMBEDDING_MODEL_NAME)
            embedding_backend = encode_and_save_minilm(text_map, embedding_prefix, _model)
            del _model
        else:
            embedding_backend = encode_and_save_tfidf(text_map, embedding_prefix)
        del text_map
        gc.collect()

    emb_train_raw = np.load(emb_file(embedding_prefix, "train")).astype(np.float32)
    emb_val_raw   = np.load(emb_file(embedding_prefix, "validation")).astype(np.float32)
    emb_test_raw  = np.load(emb_file(embedding_prefix, "test")).astype(np.float32)
    assert emb_train_raw.shape[0] == split_counts["train"]
    assert emb_val_raw.shape[0]   == split_counts["validation"]
    assert emb_test_raw.shape[0]  == split_counts["test"]

    # Free serialized strings after embedding/cache load to reduce RAM.
    for name in ["train_texts", "val_texts", "test_texts"]:
        if name in globals():
            del globals()[name]
    gc.collect()

    print(f"Embedding backend : {embedding_backend}")
    print(f"Embedding dims    : train={emb_train_raw.shape}, val={emb_val_raw.shape}, test={emb_test_raw.shape}")


load_or_create_raw_embeddings()


Loading cached embeddings [sample_100k_minilm]
Embedding backend : minilm
Embedding dims    : train=(70000, 384), val=(15000, 384), test=(15000, 384)


## 6. Tabular State Preprocessing + Embedding Reduction

In [10]:
def make_ohe(min_freq=10):
    for kwargs in [
        {"handle_unknown": "ignore", "min_frequency": min_freq, "sparse_output": True},
        {"handle_unknown": "ignore", "min_frequency": min_freq, "sparse": True},
        {"handle_unknown": "ignore", "sparse_output": True},
        {"handle_unknown": "ignore", "sparse": True},
    ]:
        try:
            return OneHotEncoder(**kwargs)
        except TypeError:
            continue
    raise RuntimeError("Cannot create sparse OneHotEncoder")


def build_preprocessor(num_cols, cat_cols):
    min_freq = 10 if RUN_MODE == "smoke" else 50
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc",  StandardScaler(with_mean=False)),
            ]), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="unknown")),
                ("ohe", make_ohe(min_freq)),
            ]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def prep_state_df(df, num_cols, cat_cols):
    out = df[num_cols + cat_cols].copy()
    for c in cat_cols:
        out[c] = out[c].where(out[c].notna(), "unknown").astype(str)
    return out


def ensure_csr(X):
    return X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(X)


preprocessor = build_preprocessor(numeric_cols, categorical_cols)

print("Fitting tabular preprocessor on train...")
tr_df = prep_state_df(train_features_df,      numeric_cols, categorical_cols)
va_df = prep_state_df(validation_features_df, numeric_cols, categorical_cols)
te_df = prep_state_df(test_features_df,       numeric_cols, categorical_cols)

X_train_sp      = ensure_csr(preprocessor.fit_transform(tr_df))
X_validation_sp = ensure_csr(preprocessor.transform(va_df))
X_test_sp       = ensure_csr(preprocessor.transform(te_df))
del tr_df, va_df, te_df
gc.collect()

del train_features_df, validation_features_df, test_features_df
gc.collect()

print(f"Sparse tabular: train={X_train_sp.shape}, val={X_validation_sp.shape}, test={X_test_sp.shape}")
print(f"Sparse density: {X_train_sp.nnz / (X_train_sp.shape[0]*X_train_sp.shape[1]):.4f}")

if not all(name in globals() for name in ["emb_train_raw", "emb_val_raw", "emb_test_raw"]):
    if "load_or_create_raw_embeddings" not in globals():
        raise RuntimeError(
            "Raw embedding arrays are missing. Run Section 5 (Local Sentence Embedding and Cache) "
            "before Section 6."
        )
    print("Raw embeddings missing in memory; loading or creating them before reduction...")
    load_or_create_raw_embeddings()

embedding_reducer_meta = None
orig_dim = emb_train_raw.shape[1]

if orig_dim > EMBEDDING_REDUCTION_COMPONENTS:
    n_comp = min(EMBEDDING_REDUCTION_COMPONENTS, orig_dim - 1, emb_train_raw.shape[0] - 1)
    print(f"Reducing embedding dim {orig_dim} → {n_comp}...")
    _sc  = StandardScaler()
    _svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
    emb_train = _svd.fit_transform(_sc.fit_transform(emb_train_raw)).astype(np.float32)
    emb_val   = _svd.transform(_sc.transform(emb_val_raw)).astype(np.float32)
    emb_test  = _svd.transform(_sc.transform(emb_test_raw)).astype(np.float32)
    embedding_reducer_meta = {"original_dim": orig_dim, "reduced_dim": n_comp,
                               "method": "StandardScaler+TruncatedSVD"}
    del _sc, _svd
else:
    emb_train = emb_train_raw
    emb_val   = emb_val_raw
    emb_test  = emb_test_raw

del emb_train_raw, emb_val_raw, emb_test_raw
gc.collect()

print(f"Embeddings (reduced): train={emb_train.shape}, val={emb_val.shape}, test={emb_test.shape}")
print(f"Reducer meta: {embedding_reducer_meta}")

state_metadata = pd.DataFrame([
    {"state": "tabular",           "split": s, "rows": X.shape[0], "cols": X.shape[1]}
    for s, X in [("train", X_train_sp), ("validation", X_validation_sp), ("test", X_test_sp)]
] + [
    {"state": "embedding_reduced", "split": s, "rows": E.shape[0], "cols": E.shape[1]}
    for s, E in [("train", emb_train), ("validation", emb_val), ("test", emb_test)]
])
state_metadata.insert(0, "sample_rows", SAMPLE_ROWS_LABEL)
state_metadata.insert(0, "run_mode", RUN_MODE)
state_metadata.to_csv(RESULTS_DIR / "phase3_state_metadata.csv", index=False)
state_metadata.to_csv(RESULTS_DIR / f"phase3_state_metadata_{RUN_OUTPUT_TAG}.csv", index=False)


Fitting tabular preprocessor on train...
Sparse tabular: train=(70000, 607), val=(15000, 607), test=(15000, 607)
Sparse density: 0.0780
Reducing embedding dim 384 → 64...
Embeddings (reduced): train=(70000, 64), val=(15000, 64), test=(15000, 64)
Reducer meta: {'original_dim': 384, 'reduced_dim': 64, 'method': 'StandardScaler+TruncatedSVD'}


## 7. Gym-style Environment and Q-value Policy

### TransactionFraudBanditEnv
| | |
|---|---|
| State `s` | feature vector for current transaction |
| Action `a` | `0 = approve`, `1 = flag/block` |
| Reward | `R = −[y*(1−a)*C_FN + (1−y)*a*C_FP]` |
| Done | True after last transaction in episode |
| `gamma` | **0** — one-step bandit, no future discounting |

**RAM key:** Env nhận sparse `X_sp` + optional dense `X_emb`.  
State per-row được **concatenate on-the-fly** → không lưu `X_train_emb` đầy đủ.

### QBanditPolicy
- Hai **`SGDRegressor`** độc lập: `Q(s, 0=approve)` và `Q(s, 1=flag)`.  
- **Quan trọng:** Mỗi policy instance chỉ nhận state với số features cố định. Không mix policy train-không-embedding với predict có-embedding.

In [11]:
# ── Cost utilities ─────────────────────────────────────────────────────────────

def compute_reward(y: int, action: int, amount: float, alpha: float, beta: float) -> float:
    c_fn = float(amount) * beta
    c_fp = float(amount) * alpha
    return -(y * (1 - action) * c_fn + (1 - y) * action * c_fp)


def fit_reward_scale_value(y, amounts, alpha, beta, mode: str) -> float:
    if mode == "raw":
        return 1.0
    y_arr = np.asarray(y, dtype=int)
    am = np.asarray(amounts, dtype=np.float32)
    potential_cost = np.where(y_arr == 1, am * float(beta), am * float(alpha)).astype(np.float32)
    positive = potential_cost[potential_cost > 0]
    if positive.size == 0:
        return 1.0
    if mode == "p95_scaled":
        return float(max(np.quantile(positive, 0.95), 1e-6))
    if mode == "log1p":
        return 1.0
    raise ValueError(f"Unsupported REWARD_SCALE_MODE={mode!r}")


def scale_training_reward(raw_reward: float, mode: str, scale_value: float) -> float:
    cost = max(0.0, -float(raw_reward))
    if mode == "raw":
        return -cost
    if mode == "p95_scaled":
        denom = max(float(scale_value), 1e-6)
        return -min(cost, denom) / denom
    if mode == "log1p":
        return -float(np.log1p(cost))
    raise ValueError(f"Unsupported reward scale mode: {mode}")


def compute_costs(y_true, actions, amounts, alpha, beta):
    y  = np.asarray(y_true,  dtype=int)
    a  = np.asarray(actions, dtype=int)
    am = np.asarray(amounts, dtype=float)
    assert set(np.unique(a)).issubset({0, 1}), "Action space must be {0, 1}"
    fn = (y == 1) & (a == 0)
    fp = (y == 0) & (a == 1)
    fn_cost = float(np.sum(am[fn] * beta))
    fp_cost = float(np.sum(am[fp] * alpha))
    return {
        "fn_cost": fn_cost, "fp_cost": fp_cost,
        "total_cost": fn_cost + fp_cost,
        "fn_count": int(fn.sum()), "fp_count": int(fp.sum()),
    }


def approve_all_cost(y_true, amounts, beta):
    return float(np.sum(np.asarray(amounts)[np.asarray(y_true) == 1] * beta))


def safe_ap(y_true, scores):
    return float(average_precision_score(y_true, scores)) if len(np.unique(y_true)) >= 2 else np.nan


def safe_roc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) >= 2 else np.nan


# ?? Gym-style environment ??????????????????????????????????????????????????????

class TransactionFraudBanditEnv:
    """
    One-step contextual bandit for fraud transaction approval.

    The business reward is always the raw negative cost. Optional reward scaling
    is applied only to the training reward returned by step(), so final metrics
    remain comparable with Phase 2 business-cost evaluation.
    """

    def __init__(self, X_sp, y, amounts, alpha, beta, X_emb=None,
                 reward_scale_mode="raw", reward_scale_value=1.0,
                 random_state=42):
        assert sparse.issparse(X_sp), "X_sp must be a scipy sparse matrix"
        self.X_sp    = X_sp.tocsr()
        self.X_emb   = X_emb
        self.y       = np.asarray(y, dtype=np.int8)
        self.amounts = np.asarray(amounts, dtype=np.float32)
        self.alpha   = float(alpha)
        self.beta    = float(beta)
        self.reward_scale_mode  = reward_scale_mode
        self.reward_scale_value = float(reward_scale_value)
        self.n       = len(y)
        self._idx    = 0
        self._order  = np.arange(self.n)
        self._rng    = np.random.RandomState(random_state)

    def reset(self, shuffle: bool = False) -> np.ndarray:
        self._idx = 0
        if shuffle:
            self._rng.shuffle(self._order)
        else:
            self._order = np.arange(self.n)
        return self._state(0)

    def step(self, action: int):
        assert action in (0, 1), f"Invalid action {action}"
        row_i = int(self._order[self._idx])
        y_i   = int(self.y[row_i])
        amt_i = float(self.amounts[row_i])
        raw_reward = compute_reward(y_i, action, amt_i, self.alpha, self.beta)
        reward = scale_training_reward(raw_reward, self.reward_scale_mode, self.reward_scale_value)
        info = {
            "y": y_i, "amount": amt_i, "action": action,
            "raw_reward": raw_reward,
            "training_reward": reward,
            "reward_scale_mode": self.reward_scale_mode,
            "reward_scale_value": self.reward_scale_value,
            "fn_cost": amt_i * self.beta  if (y_i == 1 and action == 0) else 0.0,
            "fp_cost": amt_i * self.alpha if (y_i == 0 and action == 1) else 0.0,
        }
        self._idx += 1
        done = self._idx >= self.n
        next_state = self._state(self._idx) if not done else np.zeros(self._state_dim(), dtype=np.float32)
        return next_state, reward, done, info

    def _state(self, idx) -> np.ndarray:
        row_i = int(self._order[idx])
        tab = np.asarray(self.X_sp[row_i].todense(), dtype=np.float32).flatten()
        if self.X_emb is not None:
            return np.concatenate([tab, self.X_emb[row_i].astype(np.float32)])
        return tab

    def _state_dim(self) -> int:
        d = self.X_sp.shape[1]
        if self.X_emb is not None:
            d += self.X_emb.shape[1]
        return d


# ── Q-value policy ─────────────────────────────────────────────────────────────

class QBanditPolicy:
    """
    Linear Q-learning contextual bandit.
    Two independent SGDRegressor: Q(s, approve=0) and Q(s, flag=1).
    gamma=0, online update target = immediate reward.
    Training: epsilon-greedy. Evaluation: deterministic argmax.

    IMPORTANT: Each policy instance is bound to a fixed state dimension
    (set during first partial_fit call). Do NOT mix a policy trained without
    embedding with predict calls that include embedding (or vice versa).
    """

    def __init__(self, lr: float = 0.01, random_state: int = 42):
        self._rng = np.random.RandomState(random_state)
        common = dict(learning_rate="constant", eta0=lr, penalty="l2",
                      alpha=1e-4, random_state=random_state)
        self._q = [SGDRegressor(**common), SGDRegressor(**common)]
        self._fitted = False

    def _warm_start(self, state):
        s = self._reshape(state)
        for q in self._q:
            q.partial_fit(s, [0.0])
        self._fitted = True

    @staticmethod
    def _reshape(state):
        if sparse.issparse(state):
            return state if state.shape[0] == 1 else state.reshape(1, -1)
        arr = np.asarray(state, dtype=np.float32)
        return arr.reshape(1, -1) if arr.ndim == 1 else arr

    def q_values(self, state) -> np.ndarray:
        if not self._fitted:
            return np.array([0.0, 0.0])
        s = self._reshape(state)
        return np.array([float(self._q[a].predict(s)[0]) for a in range(2)])

    def act(self, state, epsilon: float = 0.0) -> int:
        if epsilon > 0 and self._rng.random() < epsilon:
            return int(self._rng.randint(0, 2))
        return int(np.argmax(self.q_values(state)))

    def update(self, state, action: int, reward: float):
        if not self._fitted:
            self._warm_start(state)
        self._q[action].partial_fit(self._reshape(state), [float(reward)])

    def predict_scores_batched(
        self, X_sp, X_emb=None, batch_size: int = SCORE_BATCH_SIZE
    ) -> np.ndarray:
        """
        Score = Q(s,1) - Q(s,0). Higher → more likely to flag/block.
        RAM key: batch processing — never materialises full dense state matrix.
        X_sp and X_emb must match what this policy was trained on.
        """
        if not self._fitted:
            return np.zeros(X_sp.shape[0], dtype=np.float32)
        n   = X_sp.shape[0]
        out = np.empty(n, dtype=np.float32)
        for start in range(0, n, batch_size):
            end       = min(start + batch_size, n)
            batch_tab = np.asarray(X_sp[start:end].todense(), dtype=np.float32)
            if X_emb is not None:
                batch = np.concatenate([batch_tab, X_emb[start:end].astype(np.float32)], axis=1)
            else:
                batch = batch_tab
            q0 = self._q[0].predict(batch).astype(np.float32)
            q1 = self._q[1].predict(batch).astype(np.float32)
            out[start:end] = q1 - q0
            del batch_tab, batch
        return out


# ── Sanity checks ──────────────────────────────────────────────────────────────
_n_sanity = 20
_sp_dummy = sparse.random(_n_sanity, 10, density=0.5, format="csr", dtype=np.float32)
_em_dummy = np.random.randn(_n_sanity, 8).astype(np.float32)

# -- Env without embedding
_env = TransactionFraudBanditEnv(_sp_dummy, [0]*(_n_sanity-1)+[1], [100.0]*_n_sanity,
                                  alpha=0.05, beta=1.0)
_s0 = _env.reset()                      # shape (10,)
assert _s0.shape == (10,)
_s1, _r, _done, _info = _env.step(0)
assert isinstance(_r, float)
assert _info["reward_scale_mode"] == "raw"

# -- Env with embedding
_env2 = TransactionFraudBanditEnv(_sp_dummy, [1]*_n_sanity, [50.0]*_n_sanity,
                                   alpha=0.05, beta=1.0, X_emb=_em_dummy)
_s2 = _env2.reset()                     # shape (18,)
assert _s2.shape == (18,)
_s3, _r3, _, _ = _env2.step(1)
assert _r3 == 0.0

# -- Policy WITHOUT embedding: train on 10-dim states, predict on X_sp only
_pol = QBanditPolicy(lr=RL_LR)
_pol.update(_s0, 1, -5.0)
_pol.update(_s0, 0,  0.0)
assert _pol.act(_s0, epsilon=0.0) in (0, 1)
_scores = _pol.predict_scores_batched(_sp_dummy, X_emb=None, batch_size=5)
assert _scores.shape == (_n_sanity,)

# -- Policy WITH embedding: must be trained on 18-dim states (10 tab + 8 emb)
#    _pol2 is a SEPARATE instance — policy dim is fixed at first partial_fit call
_pol2 = QBanditPolicy(lr=RL_LR)
_pol2.update(_s2, 1, -5.0)             # warm-start on 18-dim state
_pol2.update(_s2, 0,  0.0)
assert _pol2.act(_s2, epsilon=0.0) in (0, 1)
_scores2 = _pol2.predict_scores_batched(_sp_dummy, X_emb=_em_dummy, batch_size=5)
assert _scores2.shape == (_n_sanity,)

del _env, _env2, _pol, _pol2, _sp_dummy, _em_dummy
del _s0, _s1, _s2, _s3, _scores, _scores2
gc.collect()
print("Sanity checks PASSED: TransactionFraudBanditEnv + QBanditPolicy")


Sanity checks PASSED: TransactionFraudBanditEnv + QBanditPolicy


## 8. Shared Evaluation Utilities

In [12]:
def tune_and_evaluate(
    model_name: str,
    scores_val: np.ndarray,
    scores_test: np.ndarray,
    cost_name: str,
    cost_config: dict,
    uses_embedding: bool,
    algorithm: str,
    extra_metadata=None,
) -> list:
    alpha = cost_config["alpha"]
    beta  = cost_config["beta"]
    extra_metadata = extra_metadata or {}

    grid = np.unique(np.concatenate([
        np.linspace(float(scores_val.min()), float(scores_val.max()), 100),
        np.quantile(scores_val, np.linspace(0.01, 0.99, 99)),
    ]))
    best_cost, best_thr = float("inf"), float(np.median(scores_val))
    for thr in grid:
        acts = (scores_val >= thr).astype(int)
        c = compute_costs(y_validation, acts, amount_validation, alpha, beta)["total_cost"]
        if c < best_cost:
            best_cost, best_thr = c, float(thr)

    rows = []
    for split_name, y, scores, amounts in [
        ("validation", y_validation, scores_val,  amount_validation),
        ("test",       y_test,       scores_test,  amount_test),
    ]:
        actions = (scores >= best_thr).astype(int)
        costs   = compute_costs(y, actions, amounts, alpha, beta)
        tn, fp, fn, tp = confusion_matrix(y, actions, labels=[0, 1]).ravel()
        app_cost = approve_all_cost(y, amounts, beta)
        rows.append({
            "run_mode":    RUN_MODE,
            "sample_rows": SAMPLE_ROWS if SAMPLE_ROWS is not None else "full",
            "model":       model_name,
            "split":       split_name,
            "cost_config": cost_name,
            "alpha": alpha, "beta": beta,
            "threshold":   best_thr,
            "pr_auc":      safe_ap(y, scores),
            "roc_auc":     safe_roc(y, scores),
            "recall_fraud":    float(recall_score(y, actions,    zero_division=0)),
            "precision_fraud": float(precision_score(y, actions, zero_division=0)),
            "f1_fraud":        float(f1_score(y, actions,        zero_division=0)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            **costs,
            "approve_all_cost":           app_cost,
            "cost_saving_vs_approve_all": app_cost - costs["total_cost"],
            "uses_embedding":   uses_embedding,
            "algorithm":        algorithm,
            "exploration":      "none_in_evaluation",
            "embedding_backend": embedding_backend if uses_embedding else "none",
            **extra_metadata,
        })
    return rows


def make_sample_weights(y, amounts, alpha, beta):
    raw = np.where(
        np.asarray(y) == 1,
        np.asarray(amounts) * beta,
        np.asarray(amounts) * alpha,
    ).astype(float)
    raw = np.maximum(raw, 1e-6)
    cap = np.quantile(raw, 0.99)
    clipped = np.minimum(raw, cap)
    return clipped / np.mean(clipped)


print("Evaluation utilities ready.")


Evaluation utilities ready.


## 9. Variant 1 — `cost_sensitive_supervised_sgd`

SGDClassifier + sample_weight — **strong supervised baseline**, NOT the main RL method.

In [13]:
all_metrics = []
training_log_rows = []
tuning_config_rows = []

print("=" * 65)
print("Variant 1: cost_sensitive_supervised_sgd")
print("=" * 65)

for cost_name, cost_cfg in COST_CONFIGS.items():
    print(f"  [{cost_name}] Fitting SGDClassifier...")
    weights = make_sample_weights(y_train, amount_train, cost_cfg["alpha"], cost_cfg["beta"])
    clf = SGDClassifier(
        loss="log_loss", penalty="l2", alpha=1e-4,
        max_iter=500, tol=1e-3, random_state=RANDOM_STATE, n_jobs=N_JOBS,
    )
    clf.fit(X_train_sp, y_train, sample_weight=weights)
    del weights

    def _sgd_scores(X):
        if hasattr(clf, "predict_proba"):
            return clf.predict_proba(X)[:, 1].astype(np.float32)
        d = clf.decision_function(X)
        return (1.0 / (1.0 + np.exp(-d))).astype(np.float32)

    sv = _sgd_scores(X_validation_sp)
    st = _sgd_scores(X_test_sp)

    rows = tune_and_evaluate(
        "cost_sensitive_supervised_sgd", sv, st, cost_name, cost_cfg,
        uses_embedding=False, algorithm="sgd_cost_sensitive_supervised",
        extra_metadata={
            "reward_scale_mode": "supervised_sample_weight",
            "reward_scale_value": np.nan,
            "rl_lr": np.nan,
            "rl_epochs": np.nan,
            "shuffle_train_each_epoch": False,
        },
    )
    all_metrics.extend(rows)
    del clf, sv, st
    gc.collect()

print("  Variant 1 DONE.")


Variant 1: cost_sensitive_supervised_sgd
  [Cost-A] Fitting SGDClassifier...
  [Cost-B] Fitting SGDClassifier...
  [Cost-C] Fitting SGDClassifier...
  Variant 1 DONE.


## 10. Variant 2 — `rl_q_bandit_without_embedding`

QBanditPolicy trained on **tabular features only** (sparse CSR).  
Predict and evaluate also use sparse — no to_dense conversion needed.

In [14]:
print("=" * 65)
print("Variant 2: rl_q_bandit_without_embedding")
print("=" * 65)

for cost_name, cost_cfg in COST_CONFIGS.items():
    alpha, beta = cost_cfg["alpha"], cost_cfg["beta"]
    reward_scale_value = fit_reward_scale_value(y_train, amount_train, alpha, beta, REWARD_SCALE_MODE)
    tuning_config_rows.append({
        "model": "rl_q_bandit_without_embedding",
        "cost_config": cost_name,
        "reward_scale_mode": REWARD_SCALE_MODE,
        "reward_scale_value": reward_scale_value,
        "rl_lr": RL_LR,
        "rl_epochs": RL_N_EPOCHS,
        "epsilon_start": RL_EPSILON_START,
        "epsilon_end": RL_EPSILON_END,
        "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
    })
    print(
        f"  [{cost_name}] Training Q-bandit ({RL_N_EPOCHS} epochs, "
        f"eps {RL_EPSILON_START}?{RL_EPSILON_END}, lr={RL_LR}, "
        f"reward={REWARD_SCALE_MODE}, scale={reward_scale_value:.4g})..."
    )

    policy = QBanditPolicy(lr=RL_LR, random_state=RANDOM_STATE)
    env = TransactionFraudBanditEnv(
        X_train_sp, y_train, amount_train,
        alpha=alpha, beta=beta, X_emb=None,
        reward_scale_mode=REWARD_SCALE_MODE,
        reward_scale_value=reward_scale_value,
        random_state=RANDOM_STATE,
    )
    epsilons = np.linspace(RL_EPSILON_START, RL_EPSILON_END, RL_N_EPOCHS)

    for epoch_idx, eps in enumerate(epsilons):
        state = env.reset(shuffle=SHUFFLE_TRAIN_EACH_EPOCH)
        cum_r, done, steps = 0.0, False, 0
        while not done:
            action = policy.act(state, epsilon=eps)
            next_state, r, done, _ = env.step(action)
            policy.update(state, action, r)
            state  = next_state
            cum_r += r
            steps += 1
        training_log_rows.append({
            "model": "rl_q_bandit_without_embedding",
            "cost_config": cost_name,
            "epoch": epoch_idx + 1,
            "epsilon": float(eps),
            "cumulative_training_reward": float(cum_r),
            "steps": int(steps),
            "reward_scale_mode": REWARD_SCALE_MODE,
            "reward_scale_value": reward_scale_value,
            "rl_lr": RL_LR,
            "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
        })
        print(f"    Epoch {epoch_idx+1}/{RL_N_EPOCHS} | eps={eps:.3f} | cumReward={cum_r:.4f}")

    _n_check = min(200, X_train_sp.shape[0])
    _qv = np.array([
        policy.q_values(np.asarray(X_train_sp[i].todense(), dtype=np.float32).flatten())
        for i in range(0, _n_check, max(1, _n_check // 50))
    ])
    assert not np.allclose(_qv[:, 0], _qv[:, 1]),         f"[{cost_name}] Q-value collapse: Q(s,0) ? Q(s,1)"
    del _qv

    sv = policy.predict_scores_batched(X_validation_sp, X_emb=None)
    st = policy.predict_scores_batched(X_test_sp,       X_emb=None)

    rows = tune_and_evaluate(
        "rl_q_bandit_without_embedding", sv, st, cost_name, cost_cfg,
        uses_embedding=False, algorithm="q_value_contextual_bandit",
        extra_metadata={
            "reward_scale_mode": REWARD_SCALE_MODE,
            "reward_scale_value": reward_scale_value,
            "rl_lr": RL_LR,
            "rl_epochs": RL_N_EPOCHS,
            "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
        },
    )
    all_metrics.extend(rows)
    del policy, env, sv, st
    gc.collect()

print("  Variant 2 DONE.")


Variant 2: rl_q_bandit_without_embedding
  [Cost-A] Training Q-bandit (3 epochs, eps 0.3?0.05, lr=0.005, reward=p95_scaled, scale=27.7)...
    Epoch 1/3 | eps=0.300 | cumReward=-7965.1979
    Epoch 2/3 | eps=0.175 | cumReward=-7806.4618
    Epoch 3/3 | eps=0.050 | cumReward=-8106.5513
  [Cost-B] Training Q-bandit (3 epochs, eps 0.3?0.05, lr=0.005, reward=p95_scaled, scale=55.4)...
    Epoch 1/3 | eps=0.300 | cumReward=-7965.1979
    Epoch 2/3 | eps=0.175 | cumReward=-7806.4618
    Epoch 3/3 | eps=0.050 | cumReward=-8106.5513
  [Cost-C] Training Q-bandit (3 epochs, eps 0.3?0.05, lr=0.005, reward=p95_scaled, scale=110.8)...
    Epoch 1/3 | eps=0.300 | cumReward=-7979.6278
    Epoch 2/3 | eps=0.175 | cumReward=-7821.2126
    Epoch 3/3 | eps=0.050 | cumReward=-8122.6484
  Variant 2 DONE.


## 11. Variant 3 — `rl_q_bandit_with_embedding`

QBanditPolicy trained on **tabular + MiniLM embedding** state (dim = d_tab + d_emb).  
State concat is on-the-fly per row in `env._state()` — no full dense matrix stored.  
Predict also uses batched concat — consistent with training dim.

In [15]:
print("=" * 65)
print("Variant 3: rl_q_bandit_with_embedding")
print("=" * 65)

for cost_name, cost_cfg in COST_CONFIGS.items():
    alpha, beta = cost_cfg["alpha"], cost_cfg["beta"]
    reward_scale_value = fit_reward_scale_value(y_train, amount_train, alpha, beta, REWARD_SCALE_MODE)
    tuning_config_rows.append({
        "model": "rl_q_bandit_with_embedding",
        "cost_config": cost_name,
        "reward_scale_mode": REWARD_SCALE_MODE,
        "reward_scale_value": reward_scale_value,
        "rl_lr": RL_LR,
        "rl_epochs": RL_N_EPOCHS,
        "epsilon_start": RL_EPSILON_START,
        "epsilon_end": RL_EPSILON_END,
        "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
    })
    print(
        f"  [{cost_name}] Training Q-bandit+emb ({RL_N_EPOCHS} epochs, "
        f"lr={RL_LR}, reward={REWARD_SCALE_MODE}, scale={reward_scale_value:.4g})..."
    )

    policy = QBanditPolicy(lr=RL_LR, random_state=RANDOM_STATE)
    env = TransactionFraudBanditEnv(
        X_train_sp, y_train, amount_train,
        alpha=alpha, beta=beta,
        X_emb=emb_train,
        reward_scale_mode=REWARD_SCALE_MODE,
        reward_scale_value=reward_scale_value,
        random_state=RANDOM_STATE,
    )
    epsilons = np.linspace(RL_EPSILON_START, RL_EPSILON_END, RL_N_EPOCHS)

    for epoch_idx, eps in enumerate(epsilons):
        state = env.reset(shuffle=SHUFFLE_TRAIN_EACH_EPOCH)
        cum_r, done, steps = 0.0, False, 0
        while not done:
            action = policy.act(state, epsilon=eps)
            next_state, r, done, _ = env.step(action)
            policy.update(state, action, r)
            state  = next_state
            cum_r += r
            steps += 1
        training_log_rows.append({
            "model": "rl_q_bandit_with_embedding",
            "cost_config": cost_name,
            "epoch": epoch_idx + 1,
            "epsilon": float(eps),
            "cumulative_training_reward": float(cum_r),
            "steps": int(steps),
            "reward_scale_mode": REWARD_SCALE_MODE,
            "reward_scale_value": reward_scale_value,
            "rl_lr": RL_LR,
            "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
        })
        print(f"    Epoch {epoch_idx+1}/{RL_N_EPOCHS} | eps={eps:.3f} | cumReward={cum_r:.4f}")

    sv = policy.predict_scores_batched(X_validation_sp, X_emb=emb_val)
    st = policy.predict_scores_batched(X_test_sp,       X_emb=emb_test)

    rows = tune_and_evaluate(
        "rl_q_bandit_with_embedding", sv, st, cost_name, cost_cfg,
        uses_embedding=True, algorithm="q_value_contextual_bandit",
        extra_metadata={
            "reward_scale_mode": REWARD_SCALE_MODE,
            "reward_scale_value": reward_scale_value,
            "rl_lr": RL_LR,
            "rl_epochs": RL_N_EPOCHS,
            "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
        },
    )
    all_metrics.extend(rows)
    del policy, env, sv, st
    gc.collect()

del emb_train, emb_val, emb_test
gc.collect()
print("  Variant 3 DONE.")


Variant 3: rl_q_bandit_with_embedding
  [Cost-A] Training Q-bandit+emb (3 epochs, lr=0.005, reward=p95_scaled, scale=27.7)...
    Epoch 1/3 | eps=0.300 | cumReward=-7779.4820
    Epoch 2/3 | eps=0.175 | cumReward=-7884.9333
    Epoch 3/3 | eps=0.050 | cumReward=-7999.9292
  [Cost-B] Training Q-bandit+emb (3 epochs, lr=0.005, reward=p95_scaled, scale=55.4)...
    Epoch 1/3 | eps=0.300 | cumReward=-7779.4820
    Epoch 2/3 | eps=0.175 | cumReward=-7884.9333
    Epoch 3/3 | eps=0.050 | cumReward=-7999.9292
  [Cost-C] Training Q-bandit+emb (3 epochs, lr=0.005, reward=p95_scaled, scale=110.8)...
    Epoch 1/3 | eps=0.300 | cumReward=-7797.1345
    Epoch 2/3 | eps=0.175 | cumReward=-7898.9354
    Epoch 3/3 | eps=0.050 | cumReward=-8013.5422
  Variant 3 DONE.


## 12. Save Results and Assertions

In [16]:
rl_ablation_df = pd.DataFrame(all_metrics)
rl_training_log_df = pd.DataFrame(training_log_rows)
rl_tuning_config_df = pd.DataFrame(tuning_config_rows)

rl_ablation_df.to_csv(RESULTS_DIR / "rl_ablation.csv", index=False)
rl_ablation_df.to_csv(RESULTS_DIR / f"rl_ablation_{RUN_OUTPUT_TAG}.csv", index=False)
rl_training_log_df.to_csv(RESULTS_DIR / "rl_training_log.csv", index=False)
rl_training_log_df.to_csv(RESULTS_DIR / f"rl_training_log_{RUN_OUTPUT_TAG}.csv", index=False)
rl_tuning_config_df.to_csv(RESULTS_DIR / "rl_tuning_config.csv", index=False)
rl_tuning_config_df.to_csv(RESULTS_DIR / f"rl_tuning_config_{RUN_OUTPUT_TAG}.csv", index=False)

REQUIRED_MODELS = {
    "cost_sensitive_supervised_sgd",
    "rl_q_bandit_without_embedding",
    "rl_q_bandit_with_embedding",
}
found_models = set(rl_ablation_df["model"].unique())
assert found_models == REQUIRED_MODELS, f"Missing models: {REQUIRED_MODELS - found_models}"
assert set(COST_CONFIGS.keys()).issubset(set(rl_ablation_df["cost_config"].unique()))

rl_rows  = rl_ablation_df[rl_ablation_df["model"].str.startswith("rl_q_bandit")]
sgd_rows = rl_ablation_df[rl_ablation_df["model"] == "cost_sensitive_supervised_sgd"]
assert (rl_rows["algorithm"]  == "q_value_contextual_bandit").all()
assert (sgd_rows["algorithm"] == "sgd_cost_sensitive_supervised").all()
assert (rl_rows["exploration"] == "none_in_evaluation").all()

print(f"rl_ablation.csv: {len(rl_ablation_df)} rows | models={sorted(found_models)}")
print(f"rl_training_log.csv: {len(rl_training_log_df)} rows")
print(f"rl_tuning_config.csv: {len(rl_tuning_config_df)} rows")
display(
    rl_ablation_df[
        ["model", "split", "cost_config", "recall_fraud",
         "precision_fraud", "total_cost", "cost_saving_vs_approve_all", "algorithm"]
    ].sort_values(["split", "cost_config", "total_cost"])
)


rl_ablation.csv: 18 rows | models=['cost_sensitive_supervised_sgd', 'rl_q_bandit_with_embedding', 'rl_q_bandit_without_embedding']
rl_training_log.csv: 18 rows
rl_tuning_config.csv: 6 rows


,model,split,cost_config,recall_fraud,precision_fraud,total_cost,cost_saving_vs_approve_all,algorithm
1,cost_sensitive_supervised_sgd,test,Cost-A,0.478689,0.100206,30349.762771,7093.272385,sgd_cost_sensitive_supervised
13,rl_q_bandit_with_embedding,test,Cost-A,0.000000,0.000000,37458.034959,-14.999803,q_value_contextual_bandit
7,rl_q_bandit_without_embedding,test,Cost-A,0.003279,0.018868,37620.627512,-177.592356,q_value_contextual_bandit
3,cost_sensitive_supervised_sgd,test,Cost-B,0.478689,0.100206,60699.525542,14186.544771,sgd_cost_sensitive_supervised
15,rl_q_bandit_with_embedding,test,Cost-B,0.000000,0.000000,74916.069919,-29.999606,q_value_contextual_bandit
9,rl_q_bandit_without_embedding,test,Cost-B,0.003279,0.018868,75241.255024,-355.184712,q_value_contextual_bandit
5,cost_sensitive_supervised_sgd,test,Cost-C,0.662295,0.064270,131006.898843,56208.288657,sgd_cost_sensitive_supervised
17,rl_q_bandit_with_embedding,test,Cost-C,0.000000,0.000000,187275.174797,-59.987297,q_value_contextual_bandit
11,rl_q_bandit_without_embedding,test,Cost-C,0.003279,0.018868,187869.111010,-653.923510,q_value_contextual_bandit
0,cost_sensitive_supervised_sgd,validation,Cost-A,0.628647,0.158635,28960.372232,16211.713706,sgd_cost_sensitive_supervised


## 13. Phase 2 Baseline Comparison + `phase3_upgrade_summary.csv`

In [17]:
SUMMARY_COLS = [
    "phase3_run_mode", "comparison_scope", "baseline_source_file",
    "run_mode", "sample_rows", "model", "split", "cost_config",
    "pr_auc", "roc_auc", "recall_fraud", "precision_fraud", "f1_fraud",
    "fn_cost", "fp_cost", "total_cost", "cost_saving_vs_approve_all",
    "uses_embedding", "algorithm", "reward_scale_mode", "reward_scale_value",
    "rl_lr", "rl_epochs", "shuffle_train_each_epoch",
]


def load_phase2_baseline_for_run():
    tagged_path = RESULTS_DIR / f"baseline_metrics_{RUN_MODE}.csv"
    canonical_path = RESULTS_DIR / "baseline_metrics.csv"
    if tagged_path.exists():
        return pd.read_csv(tagged_path), tagged_path, "matched_run_mode"
    if canonical_path.exists():
        df = pd.read_csv(canonical_path)
        if "run_mode" in df.columns:
            available_modes = set(df["run_mode"].dropna().astype(str).unique())
            if available_modes == {RUN_MODE}:
                return df, canonical_path, "matched_run_mode_canonical"
        return df, canonical_path, "mixed_scope_canonical"
    return None, None, "phase3_only"


def rel_or_empty(path):
    return str(path.relative_to(PROJECT_ROOT)) if path is not None else ""


baseline_df, baseline_path, baseline_scope = load_phase2_baseline_for_run()
phase3_test = rl_ablation_df[rl_ablation_df["split"] == "test"].copy()
phase3_test["phase3_run_mode"] = RUN_MODE
phase3_test["comparison_scope"] = baseline_scope
phase3_test["baseline_source_file"] = rel_or_empty(baseline_path)

if baseline_df is not None:
    baseline_test = baseline_df[baseline_df["split"] == "test"].copy()
    if "run_mode" not in baseline_test.columns:
        baseline_test.insert(0, "run_mode", "phase2_unknown")
    if "sample_rows" not in baseline_test.columns:
        baseline_test["sample_rows"] = "unknown"
    if "uses_embedding" not in baseline_test.columns:
        baseline_test["uses_embedding"] = False
    if "algorithm" not in baseline_test.columns:
        baseline_test["algorithm"] = "supervised_ml_baseline"
    baseline_test["phase3_run_mode"] = RUN_MODE
    baseline_test["comparison_scope"] = baseline_scope
    baseline_test["baseline_source_file"] = rel_or_empty(baseline_path)

    base_sub = baseline_test[[c for c in SUMMARY_COLS if c in baseline_test.columns]]
    phase3_sub = phase3_test[[c for c in SUMMARY_COLS if c in phase3_test.columns]]
    upgrade_summary = pd.concat([base_sub, phase3_sub], ignore_index=True, sort=False)
    print(f"Phase 2 + Phase 3 summary loaded from {rel_or_empty(baseline_path)} [{baseline_scope}]")
    if baseline_scope.startswith("mixed_scope"):
        print("WARNING: Phase 2 baseline is not sample-matched with this Phase 3 run. Do not use this table for superiority claims.")
else:
    upgrade_summary = phase3_test[[c for c in SUMMARY_COLS if c in phase3_test.columns]]
    print("No Phase 2 baseline found. Saving Phase 3 results only.")

upgrade_summary.to_csv(RESULTS_DIR / "phase3_upgrade_summary.csv", index=False)
upgrade_summary.to_csv(RESULTS_DIR / "phase3_comparison_preview.csv", index=False)
upgrade_summary.to_csv(RESULTS_DIR / f"phase3_upgrade_summary_{RUN_OUTPUT_TAG}.csv", index=False)
upgrade_summary.to_csv(RESULTS_DIR / f"phase3_comparison_preview_{RUN_OUTPUT_TAG}.csv", index=False)
display(upgrade_summary.sort_values(["cost_config", "total_cost"]).head(24))


Phase 2 + Phase 3 summary loaded from results/baseline_metrics_sample_100k.csv [matched_run_mode]


,phase3_run_mode,comparison_scope,baseline_source_file,run_mode,sample_rows,model,split,cost_config,pr_auc,roc_auc,recall_fraud,precision_fraud,f1_fraud,fn_cost,fp_cost,total_cost,cost_saving_vs_approve_all,uses_embedding,algorithm,reward_scale_mode,reward_scale_value,rl_lr,rl_epochs,shuffle_train_each_epoch
2,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,lightgbm_balanced,test,Cost-A,0.460326,0.886567,0.567213,0.335271,0.421437,18039.317970,2159.069448,20198.387418,17244.647541,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
3,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,xgboost_magic_style,test,Cost-A,0.437764,0.891059,0.632787,0.143601,0.234081,16389.751976,8212.302143,24602.054119,12840.980840,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
12,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,cost_sensitive_supervised_sgd,test,Cost-A,0.108053,0.783033,0.478689,0.100206,0.165721,26116.406976,4233.355795,30349.762771,7093.272385,False,sgd_cost_sensitive_supervised,supervised_sample_weight,NaN,NaN,NaN,False
1,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,logistic_regression_balanced,test,Cost-A,0.239638,0.797354,0.400000,0.239686,0.299754,26982.424971,4041.330498,31023.755469,6419.279491,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
0,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,approve_all,test,Cost-A,0.020333,0.500000,0.000000,0.000000,0.000000,37443.034959,0.000000,37443.034959,0.000000,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
18,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,rl_q_bandit_with_embedding,test,Cost-A,0.015520,0.381151,0.000000,0.000000,0.000000,37443.034959,15.000000,37458.034959,-14.999803,True,q_value_contextual_bandit,p95_scaled,27.700001,0.005,3.0,False
15,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,rl_q_bandit_without_embedding,test,Cost-A,0.019764,0.483001,0.003279,0.018868,0.005587,37386.600961,234.026551,37620.627512,-177.592356,False,q_value_contextual_bandit,p95_scaled,27.700001,0.005,3.0,False
6,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,lightgbm_balanced,test,Cost-B,0.460326,0.886567,0.567213,0.335271,0.421437,36078.635940,4318.138897,40396.774837,34489.295082,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
7,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,xgboost_magic_style,test,Cost-B,0.437764,0.891059,0.632787,0.143601,0.234081,32779.503953,16424.604285,49204.108238,25681.961680,False,supervised_ml_baseline,NaN,NaN,NaN,NaN,NaN
13,sample_100k,matched_run_mode,results/baseline_metrics_sample_100k.csv,sample_100k,100000,cost_sensitive_supervised_sgd,test,Cost-B,0.108053,0.783033,0.478689,0.100206,0.165721,52232.813951,8466.711590,60699.525542,14186.544771,False,sgd_cost_sensitive_supervised,supervised_sample_weight,NaN,NaN,NaN,False


## 14. Figures

In [18]:
def save_fig(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()


test_rows = rl_ablation_df[rl_ablation_df["split"] == "test"]
MODEL_PALETTE = {
    "cost_sensitive_supervised_sgd":  "#4c72b0",
    "rl_q_bandit_without_embedding":  "#dd8452",
    "rl_q_bandit_with_embedding":     "#55a868",
}

cb = test_rows[test_rows["cost_config"] == "Cost-B"].copy()

plt.figure(figsize=(9, 5))
sns.barplot(data=cb, x="model", y="total_cost", palette=MODEL_PALETTE, order=list(MODEL_PALETTE))
plt.xticks(rotation=15, ha="right")
plt.title("Phase 3 — Total Cost (Cost-B, Test Split)")
plt.xlabel("Model")
plt.ylabel("Total Cost")
save_fig(FIGURES_DIR / "phase3_total_cost_cost_b.png")

plt.figure(figsize=(9, 5))
sns.barplot(data=cb, x="model", y="cost_saving_vs_approve_all",
            palette=MODEL_PALETTE, order=list(MODEL_PALETTE))
plt.axhline(0, color="red", linestyle="--", linewidth=1, label="Approve-all baseline")
plt.xticks(rotation=15, ha="right")
plt.title("Phase 3 — Cost Saving vs Approve-All (Cost-B, Test)")
plt.xlabel("Model")
plt.ylabel("Cost Saving")
plt.legend()
save_fig(FIGURES_DIR / "phase3_cost_saving_cost_b.png")

plt.figure(figsize=(7, 5))
for model_name, grp in test_rows.groupby("model"):
    plt.scatter(grp["recall_fraud"], grp["precision_fraud"],
                label=model_name, s=70, alpha=0.85, color=MODEL_PALETTE.get(model_name))
    for _, row in grp.iterrows():
        plt.annotate(row["cost_config"],
                     (row["recall_fraud"], row["precision_fraud"]), fontsize=7, alpha=0.7)
plt.xlabel("Recall Fraud")
plt.ylabel("Precision Fraud")
plt.title("Phase 3 — Recall vs Precision (Test, all cost configs)")
plt.legend(fontsize=8)
save_fig(FIGURES_DIR / "phase3_recall_precision_scatter.png")

pivot = test_rows.pivot_table(
    index="model", columns="cost_config", values="total_cost", aggfunc="mean"
)
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5)
plt.title("Phase 3 — Total Cost Heatmap (Test Split)")
plt.ylabel("Model")
plt.xlabel("Cost Config")
save_fig(FIGURES_DIR / "phase3_cost_heatmap_test.png")

del cb, test_rows
gc.collect()
print(f"Figures saved to {FIGURES_DIR}")

Figures saved to /content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive/reports/figures


## 15. Run Metadata and Handoff

In [19]:
run_metadata = {
    "phase":   3,
    "version": "upgraded_q_bandit_2026-05-27",
    "run_mode":  RUN_MODE,
    "sample_rows": SAMPLE_ROWS_LABEL,
    "run_output_tag": RUN_OUTPUT_TAG,
    "random_state": RANDOM_STATE,
    "project_root": str(PROJECT_ROOT),
    "embedding_model_target": EMBEDDING_MODEL_NAME,
    "embedding_backend_used": embedding_backend,
    "embedding_prefix":       embedding_prefix,
    "embedding_reducer":      embedding_reducer_meta,
    "cost_configs": COST_CONFIGS,
    "phase2_baseline_source": rel_or_empty(baseline_path),
    "phase2_baseline_comparison_scope": baseline_scope,
    "rl_hyperparams": {
        "n_epochs":       RL_N_EPOCHS,
        "epsilon_start":  RL_EPSILON_START,
        "epsilon_end":    RL_EPSILON_END,
        "learning_rate":  RL_LR,
        "reward_scale_mode": REWARD_SCALE_MODE,
        "shuffle_train_each_epoch": SHUFFLE_TRAIN_EACH_EPOCH,
        "run_q_bandit_tuning_grid": RUN_Q_BANDIT_TUNING_GRID,
        "q_bandit_tuning_grid": Q_BANDIT_TUNING_GRID,
        "gamma":          0,
        "q_model":        "2 x sklearn.SGDRegressor (one per action)",
        "state_v2":       "sparse_tabular_row only",
        "state_v3":       "sparse_tabular_row || emb_row (on-the-fly, no full matrix)",
    },
    "ram_optimizations": [
        "usecols on train_transaction.csv",
        "identity_df deleted after join",
        "text serialization in chunks (TEXT_SERIALIZE_CHUNK rows)",
        "text strings deleted after embedding encode",
        "embeddings saved split-by-split during encode",
        "tabular state kept as sparse CSR",
        "Variant 2: SGDRegressor receives sparse natively",
        "Variant 3: state concat on-the-fly per row in env._state()",
        "predict_scores_batched: SCORE_BATCH_SIZE rows at a time",
        "del+gc.collect() after each major step",
        "float32 / int8 for all numeric arrays",
        "policy dim fixed at first partial_fit — separate instances per variant",
    ],
    "models": {
        "cost_sensitive_supervised_sgd":  "SGDClassifier + sample_weight (supervised baseline)",
        "rl_q_bandit_without_embedding":  "QBanditPolicy - tabular only, no embedding",
        "rl_q_bandit_with_embedding":     "QBanditPolicy - tabular + MiniLM embedding",
    },
    "split_rows": {name: int(len(df)) for name, df in splits.items()},
    "text_columns":              text_cols,
    "numeric_state_columns":     numeric_cols,
    "categorical_state_columns": categorical_cols,
    "outputs": {
        "rl_ablation":            str((RESULTS_DIR / "rl_ablation.csv").relative_to(PROJECT_ROOT)),
        "rl_ablation_tagged":     str((RESULTS_DIR / f"rl_ablation_{RUN_OUTPUT_TAG}.csv").relative_to(PROJECT_ROOT)),
        "rl_training_log_tagged": str((RESULTS_DIR / f"rl_training_log_{RUN_OUTPUT_TAG}.csv").relative_to(PROJECT_ROOT)),
        "rl_tuning_config_tagged": str((RESULTS_DIR / f"rl_tuning_config_{RUN_OUTPUT_TAG}.csv").relative_to(PROJECT_ROOT)),
        "phase3_upgrade_summary": str((RESULTS_DIR / "phase3_upgrade_summary.csv").relative_to(PROJECT_ROOT)),
        "phase3_upgrade_summary_tagged": str((RESULTS_DIR / f"phase3_upgrade_summary_{RUN_OUTPUT_TAG}.csv").relative_to(PROJECT_ROOT)),
        "embedding_metadata":     str(meta_file(embedding_prefix).relative_to(PROJECT_ROOT)),
    },
    "notes": (
        "Smoke results are pipeline verification only. Do not use for Phase 4 claims."
        if RUN_MODE == "smoke"
        else (
            f"{RUN_MODE} run. Compare only with Phase 2 {RUN_MODE} outputs, or mark mixed-scope explicitly."
            if RUN_MODE.startswith("sample_")
            else "Full run output. Validate embedding_backend_used=minilm before Phase 4 analysis."
        )
    ),
}

for metadata_path in [
    RESULTS_DIR / "phase3_run_metadata.json",
    RESULTS_DIR / f"phase3_run_metadata_{RUN_OUTPUT_TAG}.json",
]:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(run_metadata, f, ensure_ascii=False, indent=2)

completion_summary = pd.DataFrame([
    {"check": "text_leakage_audit",             "status": "PASS"},
    {"check": "embedding_cache",                "status": "PASS", "backend": embedding_backend},
    {"check": "cost_sensitive_supervised_sgd",  "status": "PASS"},
    {"check": "rl_q_bandit_without_embedding",  "status": "PASS"},
    {"check": "rl_q_bandit_with_embedding",     "status": "PASS"},
    {"check": "env_reset_step_verified",        "status": "PASS"},
    {"check": "q_value_divergence_verified",    "status": "PASS"},
    {"check": "evaluation_no_exploration",      "status": "PASS"},
    {"check": "phase3_upgrade_summary_csv",     "status": "PASS"},
    {"check": "run_mode",                       "status": RUN_MODE},
    {"check": "sample_rows",                    "status": SAMPLE_ROWS_LABEL},
    {"check": "comparison_scope",               "status": baseline_scope},
    {"check": "reward_scale_mode",              "status": REWARD_SCALE_MODE},
])
completion_summary.to_csv(RESULTS_DIR / "phase3_completion_summary.csv", index=False)
completion_summary.to_csv(RESULTS_DIR / f"phase3_completion_summary_{RUN_OUTPUT_TAG}.csv", index=False)

print("=" * 65)
print("Phase 3 Upgrade — Completion Summary")
print("=" * 65)
display(completion_summary)
print("\nRun metadata (excerpt):")
print(json.dumps(run_metadata, ensure_ascii=False, indent=2)[:2500])


Phase 3 Upgrade — Completion Summary


,check,status,backend
0,text_leakage_audit,PASS,NaN
1,embedding_cache,PASS,minilm
2,cost_sensitive_supervised_sgd,PASS,NaN
3,rl_q_bandit_without_embedding,PASS,NaN
4,rl_q_bandit_with_embedding,PASS,NaN
5,env_reset_step_verified,PASS,NaN
6,q_value_divergence_verified,PASS,NaN
7,evaluation_no_exploration,PASS,NaN
8,phase3_upgrade_summary_csv,PASS,NaN
9,run_mode,sample_100k,NaN



Run metadata (excerpt):
{
  "phase": 3,
  "version": "upgraded_q_bandit_2026-05-27",
  "run_mode": "sample_100k",
  "sample_rows": 100000,
  "run_output_tag": "sample_100k",
  "random_state": 42,
  "project_root": "/content/drive/MyDrive/BaoMatCuoiKy/LLM-Assisted_Cost-Sensitive",
  "embedding_model_target": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_backend_used": "minilm",
  "embedding_prefix": "sample_100k_minilm",
  "embedding_reducer": {
    "original_dim": 384,
    "reduced_dim": 64,
    "method": "StandardScaler+TruncatedSVD"
  },
  "cost_configs": {
    "Cost-A": {
      "alpha": 0.05,
      "beta": 1.0
    },
    "Cost-B": {
      "alpha": 0.1,
      "beta": 2.0
    },
    "Cost-C": {
      "alpha": 0.2,
      "beta": 5.0
    }
  },
  "phase2_baseline_source": "results/baseline_metrics_sample_100k.csv",
  "phase2_baseline_comparison_scope": "matched_run_mode",
  "rl_hyperparams": {
    "n_epochs": 3,
    "epsilon_start": 0.3,
    "epsilon_end": 0.05,
    "learning_

## 16. Phase 3 Conclusion

| Thành phần | Mô tả |
|---|---|
| `TransactionFraudBanditEnv` | `reset()`, `step(action)`, reward `= −cost`, done, info |
| `QBanditPolicy` | 2 SGDRegressor, `gamma=0`, epsilon-greedy train, deterministic eval |
| `cost_sensitive_supervised_sgd` | SGDClassifier + sample_weight — supervised baseline |
| `rl_q_bandit_without_embedding` | Policy dim = d_tab; predict `X_emb=None` |
| `rl_q_bandit_with_embedding` | Policy dim = d_tab + d_emb; predict `X_emb=emb_val/test` |
| Policy dim safety | Separate instances per variant — no cross-dim contamination |
| Colab/Drive setup | Cell 0: mount Drive, `PROJECT_ROOT_OVERRIDE`, auto-detect |
| RAM: `usecols` | Chỉ load cột cần thiết từ transaction CSV |
| RAM: batch serialize | Text theo `TEXT_SERIALIZE_CHUNK` rows |
| RAM: sparse env | Training loop không materialized full dense state |
| RAM: batched scores | `SCORE_BATCH_SIZE` rows/lần khi predict |

**Bước tiếp theo:**
1. Chạy notebook 02 và 03 với cùng `RUN_MODE = "sample_100k"`.
2. Nếu ổn, tăng dần cả hai notebook sang `"sample_200k"`, rồi `"sample_300k"`.
3. Chỉ đổi sang `"full"` khi Colab còn đủ RAM/runtime.
4. Xác nhận `embedding_backend_used = "minilm"` trong metadata.
5. Tải `results/` về local, bắt đầu Phase 4.
